<a href="https://colab.research.google.com/github/mdsadaqathali/week12/blob/main/w12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

data=pd.read_csv("resume_data_for_ranking.csv")

print("Dataset loaded successfully")
print("Total records:",len(data))

encoder=SentenceTransformer("all-MiniLM-L6-v2")

def clean_value(value):
    if pd.isna(value):
        return ""

    if isinstance(value,list):
        return " ".join(map(str,value))

    try:
        value=ast.literal_eval(str(value))

        if isinstance(value,list):
            return " ".join(map(str,value))
    except:
        pass

    return str(value)

columns=[
    "skills",
    "career_objective",
    "experiencere_requirement",
    "job_position_name",
    "skills_required",
    "responsibilities.1"
]

for column in columns:
    data[column+"_clean"]=data[column].apply(clean_value)

def build_resume(row):
    return " ".join([
        row["career_objective_clean"],
        row["skills_clean"],
        row["experiencere_requirement_clean"]
    ])

def build_job(row):
    education=clean_value(row["educationaL_requirements"])

    return " ".join([
        row["job_position_name_clean"],
        row["skills_required_clean"],
        row["responsibilities.1_clean"],
        education
    ])

def split_skills(text):
    return set(
        item.strip().lower()
        for item in text.split(",")
        if item.strip()
    )

def find_experience(text):
    text=str(text).lower()

    digits=""

    for character in text:
        if character.isdigit():
            digits+=character
        elif digits:
            return int(digits)

    if digits:
        return int(digits)

    return 0

screening_results=[]

for number,row in data.head(10).iterrows():

    resume=build_resume(row)
    job=build_job(row)

    resume_vector=encoder.encode(resume)
    job_vector=encoder.encode(job)

    similarity=cosine_similarity(
        [resume_vector],
        [job_vector]
    )[0][0]

    candidate_skills=split_skills(
        row["skills_clean"]
    )

    required_skills=split_skills(
        row["skills_required_clean"]
    )

    matched_skills=candidate_skills & required_skills
    missing_skills=required_skills-candidate_skills

    if required_skills:
        skill_score=len(matched_skills)/len(required_skills)
    else:
        skill_score=0

    experience=find_experience(
        row["experiencere_requirement_clean"]
    )

    if experience>0:
        experience_score=1
    else:
        experience_score=0.5

    total_score=(
        0.4*skill_score+
        0.4*similarity+
        0.2*experience_score
    )

    match_percentage=round(
        total_score*100,
        2
    )

    if match_percentage>=75:
        result="Strong Candidate"
    elif match_percentage>=50:
        result="Moderate Candidate"
    else:
        result="Weak Candidate"

    screening_results.append({
        "Candidate":"Candidate "+str(number+1),
        "Job Position":row["job_position_name"],
        "Match Score":match_percentage,
        "Matched Skills":", ".join(sorted(matched_skills)),
        "Missing Skills":", ".join(sorted(missing_skills)),
        "Recommendation":result
    })

output=pd.DataFrame(screening_results)

print("\nINTELLIGENT RESUME SCREENING RESULTS")
print(output.to_string(index=False))

print("\nAverage Match Score:",
      round(output["Match Score"].mean(),2))

Dataset loaded successfully
Total records: 9544


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


INTELLIGENT RESUME SCREENING RESULTS
   Candidate                                                   Job Position  Match Score Matched Skills                                                                                                                       Missing Skills Recommendation
 Candidate 1                                       Senior Software Engineer    32.200001                                                                                                                                                     Weak Candidate
 Candidate 2                                 Machine Learning (ML) Engineer    41.930000                                                                                                                                                     Weak Candidate
 Candidate 3 Executive/ Senior Executive- Trade Marketing, Hygiene Products    24.040001                                     brand promotion\ncampaign management\nfield supervision\nmerchandising\npromotion